# RQ5. Sensitivity to Evaluation Metrics

This notebook answers RQ5 by ranking models separately by MAE, RMSE, and R².

**Dataset:** Kaggle AI and Data Science Job Market Dataset 2020 to 2026.

**Target variable:** `salary`

**Task type:** Supervised learning regression

**Outputs:** This notebook saves its result table as a CSV file and its publication ready figure as a PDF file under `/kaggle/working/ai_job_market_outputs/`.

## 1. Setup and reusable functions

Run this cell first. It automatically searches `/kaggle/input` for a CSV or Excel file containing the expected AI job market columns. If needed, set `DATA_PATH` manually inside the setup cell.

In [ ]:

# Common setup: dataset loading, preprocessing, and evaluation utilities
import os
import glob
import time
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, KFold, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

RANDOM_STATE = 42
TARGET_COL = "salary"
ID_COL = "job_id"
OUTPUT_DIR = "/kaggle/working/ai_job_market_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

plt.rcParams.update({
    "figure.dpi": 300,
    "savefig.dpi": 300,
    "font.size": 10,
    "axes.titlesize": 12,
    "axes.labelsize": 10,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "legend.fontsize": 9,
    "figure.titlesize": 13,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

# Optional manual path. Leave as None on Kaggle unless you want to force a specific file.
DATA_PATH = None

def clean_column_names(df):
    df = df.copy()
    df.columns = (
        df.columns.astype(str)
        .str.strip()
        .str.lower()
        .str.replace(" ", "_", regex=False)
        .str.replace("-", "_", regex=False)
        .str.replace("/", "_", regex=False)
    )
    return df

def read_any_table(path):
    ext = os.path.splitext(path)[1].lower()
    if ext == ".csv":
        return pd.read_csv(path)
    if ext in [".xlsx", ".xls"]:
        return pd.read_excel(path)
    raise ValueError(f"Unsupported file type: {path}")

def find_dataset_file():
    if DATA_PATH is not None:
        return DATA_PATH
    patterns = [
        "/kaggle/input/**/*.csv",
        "/kaggle/input/**/*.xlsx",
        "/kaggle/input/**/*.xls",
        "./*.csv",
        "./*.xlsx",
        "./*.xls",
    ]
    candidates = []
    for pattern in patterns:
        candidates.extend(glob.glob(pattern, recursive=True))
    candidates = sorted(set(candidates))
    if not candidates:
        raise FileNotFoundError("No CSV or Excel file found. Upload the Kaggle dataset or set DATA_PATH manually.")
    # Prefer the file that contains the expected target and job market columns.
    scored = []
    expected = {"salary", "job_title", "experience_level", "years_experience"}
    for path in candidates:
        try:
            sample = read_any_table(path).head(5)
            cols = set(clean_column_names(sample).columns)
            score = len(expected.intersection(cols))
            name_bonus = sum(token in os.path.basename(path).lower() for token in ["ai", "job", "market", "data"])
            scored.append((score + 0.1 * name_bonus, path))
        except Exception:
            continue
    if not scored:
        raise FileNotFoundError("Files were found, but none could be read as a valid CSV or Excel dataset.")
    scored.sort(reverse=True)
    return scored[0][1]

def load_dataset():
    path = find_dataset_file()
    df = read_any_table(path)
    df = clean_column_names(df)
    print(f"Loaded dataset: {path}")
    print(f"Shape: {df.shape}")
    if TARGET_COL not in df.columns:
        raise ValueError(f"Target column '{TARGET_COL}' was not found. Available columns: {df.columns.tolist()}")
    return df

def split_features_target(df):
    df = df.copy()
    # Keep only rows with observed salary.
    df = df.dropna(subset=[TARGET_COL])
    y = df[TARGET_COL].astype(float)
    drop_cols = [TARGET_COL]
    if ID_COL in df.columns:
        drop_cols.append(ID_COL)
    X = df.drop(columns=drop_cols)
    return X, y

def make_one_hot_encoder():
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)

def get_feature_groups(X):
    categorical_features = X.select_dtypes(include=["object", "category", "bool"]).columns.tolist()
    numerical_features = [c for c in X.columns if c not in categorical_features]
    return numerical_features, categorical_features

def make_preprocessor(X, scale_numeric=False):
    numerical_features, categorical_features = get_feature_groups(X)
    if scale_numeric:
        numeric_transformer = Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler())
        ])
    else:
        numeric_transformer = Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="median"))
        ])
    categorical_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", make_one_hot_encoder())
    ])
    preprocessor = ColumnTransformer(
        transformers=[
            ("num", numeric_transformer, numerical_features),
            ("cat", categorical_transformer, categorical_features),
        ],
        remainder="drop",
        sparse_threshold=0.0,
    )
    return preprocessor

def evaluate_predictions(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    return mae, rmse, r2

def evaluate_model(name, model, X_train, X_test, y_train, y_test, scale_numeric=False):
    pipe = Pipeline(steps=[
        ("preprocessor", make_preprocessor(X_train, scale_numeric=scale_numeric)),
        ("model", model)
    ])
    start = time.time()
    pipe.fit(X_train, y_train)
    fit_time = time.time() - start
    pred_start = time.time()
    y_pred = pipe.predict(X_test)
    pred_time = time.time() - pred_start
    mae, rmse, r2 = evaluate_predictions(y_test, y_pred)
    return {
        "Model": name,
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2,
        "Fit_Time_Seconds": fit_time,
        "Prediction_Time_Seconds": pred_time,
        "Pipeline": pipe
    }

def save_table(df, filename):
    path = os.path.join(OUTPUT_DIR, filename)
    df.to_csv(path, index=False)
    print(f"Saved table: {path}")
    return path

def save_figure(filename):
    path = os.path.join(OUTPUT_DIR, filename)
    plt.savefig(path, format="pdf", bbox_inches="tight")
    print(f"Saved figure: {path}")
    return path


## 2. Run the experiment for this research question

This section loads the raw dataset, trains the required model or models, creates the actual result table, and exports the corresponding figure.

In [ ]:

from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import Ridge

# XGBoost is intentionally not required so the notebook runs quickly on Kaggle CPU.
HAS_XGB = False

# Load and split data
df = load_dataset()
X, y = split_features_target(df)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=RANDOM_STATE)

models = [
    ("Linear Regression", LinearRegression(), True),
    ("Ridge Regression", Ridge(alpha=1.0), True),
    ("Decision Tree", DecisionTreeRegressor(random_state=RANDOM_STATE, max_depth=12), False),
    ("Random Forest", RandomForestRegressor(n_estimators=60, random_state=RANDOM_STATE, n_jobs=-1, max_depth=18), False),
    ("Gradient Boosting", GradientBoostingRegressor(random_state=RANDOM_STATE, n_estimators=60), False),
]
if HAS_XGB:
    models.append(("XGBoost", XGBRegressor(n_estimators=300, learning_rate=0.05, max_depth=5, random_state=RANDOM_STATE, objective="reg:squarederror", n_jobs=-1), False))

results = []
for name, model, scale_numeric in models:
    out = evaluate_model(name, model, X_train, X_test, y_train, y_test, scale_numeric=scale_numeric)
    results.append({
        "Model": name,
        "MAE": out["MAE"],
        "RMSE": out["RMSE"],
        "R2": out["R2"]
    })

performance = pd.DataFrame(results)
ranking = performance[["Model", "MAE", "RMSE", "R2"]].copy()
ranking["Rank_by_MAE"] = ranking["MAE"].rank(method="min", ascending=True).astype(int)
ranking["Rank_by_RMSE"] = ranking["RMSE"].rank(method="min", ascending=True).astype(int)
ranking["Rank_by_R2"] = ranking["R2"].rank(method="min", ascending=False).astype(int)
ranking["Average_Rank"] = ranking[["Rank_by_MAE", "Rank_by_RMSE", "Rank_by_R2"]].mean(axis=1)
ranking = ranking.sort_values("Average_Rank")
print(ranking)
save_table(ranking, "RQ5_metric_sensitivity_ranking.csv")

# Figure 5: model rank changes across metrics
plot_df = ranking.copy()
metrics = ["Rank_by_MAE", "Rank_by_RMSE", "Rank_by_R2"]
metric_labels = ["MAE", "RMSE", "R²"]

fig, ax = plt.subplots(figsize=(8, 5))
x = np.arange(len(metrics))
for _, row in plot_df.iterrows():
    ax.plot(x, [row[m] for m in metrics], marker="o", linewidth=2, label=row["Model"])
ax.set_xticks(x)
ax.set_xticklabels(metric_labels)
ax.set_ylabel("Model rank")
ax.set_title("Figure 5. Sensitivity of Model Ranking to Evaluation Metrics")
ax.set_ylim(plot_df[metrics].max().max() + 0.5, 0.5)
ax.set_yticks(range(1, int(plot_df[metrics].max().max()) + 1))
ax.legend(loc="center left", bbox_to_anchor=(1.02, 0.5), frameon=False)
plt.tight_layout()
save_figure("RQ5_metric_sensitivity_ranking.pdf")
